In [1]:
import os
from pyspark.sql.functions import count, col, when, broadcast, udf, pandas_udf, rand, monotonically_increasing_id
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, PCA, StandardScalerModel, PCAModel
from pyspark.ml.clustering import KMeans, KMeansModel
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, BooleanType, DoubleType, ArrayType, IntegerType, StructType, StructField, FloatType
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from pyspark.ml.evaluation import ClusteringEvaluator
import datamol as dm
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
import logging
import matplotlib.pyplot as plt
import findspark
import os
import sys

In [14]:
spark = SparkSession.builder\
    .appName("Molecule Analysis")\
    .config("spark.executor.memory", "4g")\
    .config("spark.driver.memory", "4g")\
    .config("spark.memory.fraction", "0.7")\
    .config("spark.memoary.storageFraction", "0.4")\
    .config("spark.network.timeout", "1200s")\
    .config("spark.executor.heartbeatInterval", "200s")\
    .config("spark.sql.broadcastTimeout", "1500s")\
    .getOrCreate()


25/02/07 14:04:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [15]:
filename = "leash-BELKA/train.parquet"
data = spark.read.parquet(filename)

data = data.filter(col("protein_name") == "sEH")
sampled_data = data.orderBy(rand()).limit(100)

In [16]:
desc_schema = StructType([
    StructField("mol_wt", DoubleType(), True),
    StructField("mol_logp", DoubleType(), True),
    StructField("tpsa", DoubleType(), True),
]
)

@pandas_udf(desc_schema)
def calculate_desc(smiles_series: pd.Series) -> pd.DataFrame:

    mols = smiles_series.apply(Chem.MolFromSmiles)
    
    mw = mols.apply(lambda mol: Descriptors.MolWt(mol) if mol else None)
    logp = mols.apply(lambda mol: Descriptors.MolLogP(mol) if mol else None)
    tpsa = mols.apply(lambda mol: Descriptors.TPSA(mol) if mol else None)

    return pd.DataFrame({'mol_wt': mw, 'mol_logp': logp, 'tpsa': tpsa})
    
sampled_data_with_desc = sampled_data.withColumn("molecule", calculate_desc(col("molecule_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block1", calculate_desc(col("buildingblock1_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block2", calculate_desc(col("buildingblock2_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block3", calculate_desc(col("buildingblock3_smiles")))

In [17]:
selected_columns = [
    "id",
    col("molecule.mol_wt").alias("molecule_mol_wt"),
    col("molecule.mol_logp").alias("molecule_mol_logp"),
    col("molecule.tpsa").alias("molecule_tpsa"),
    col("block1.mol_wt").alias("block1_mol_wt"),
    col("block1.mol_logp").alias("block1_mol_logp"),
    col("block1.tpsa").alias("block1_tpsa"),
    col("block2.mol_wt").alias("block2_mol_wt"),
    col("block2.mol_logp").alias("block2_mol_logp"),
    col("block2.tpsa").alias("block2_tpsa"),
    col("block3.mol_wt").alias("block3_mol_wt"),
    col("block3.mol_logp").alias("block3_mol_logp"),
    col("block3.tpsa").alias("block3_tpsa"),
    "binds"
]

flattened_data = sampled_data_with_desc.select(*selected_columns)

In [18]:
desc_columns = [col_name for col_name in flattened_data.columns if col_name != "id" and col_name != "binds"]

assembler = VectorAssembler(inputCols=desc_columns, outputCol="features")
sampled_data_with_features = assembler.transform(flattened_data)

In [ ]:
spark.stop()

In [19]:
scaler_model = StandardScalerModel.load("./intermediates/scaler_model")
scaled_data = scaler_model.transform(sampled_data_with_features)

In [20]:
pca_model = PCAModel.load("./intermediates/pca_model")
reduced_data = pca_model.transform(scaled_data)

In [21]:
kmeans_model = KMeansModel.load("./intermediates/kmeans_model")
kmeans_clusters = kmeans_model.transform(reduced_data).select("id", "binds", "kmeans_cluster")

In [22]:
binds1 = kmeans_clusters.filter(col("binds") == 1)
binds0 = kmeans_clusters.filter(col("binds") == 0)

In [23]:
binds_1_counts = binds1.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_1")
binds_0_counts = binds0.groupBy("kmeans_cluster").count().withColumnRenamed("count", "count_binds_0")


cluster_counts = binds_1_counts.join(binds_0_counts, "kmeans_cluster", "inner")

In [25]:
cluster_counts.show()

+--------------+-------------+-------------+
|kmeans_cluster|count_binds_1|count_binds_0|
+--------------+-------------+-------------+
+--------------+-------------+-------------+



In [24]:
fractions_df = cluster_counts.withColumn("fraction", (F.col("count_binds_1") * 3) / F.col("count_binds_0"))
fractions_df.show()

+--------------+-------------+-------------+--------+
|kmeans_cluster|count_binds_1|count_binds_0|fraction|
+--------------+-------------+-------------+--------+
+--------------+-------------+-------------+--------+



In [12]:
fractions_df = cluster_counts.withColumn("fraction", (F.col("count_binds_1") * 3) / F.col("count_binds_0"))

fractions = fractions_df.select("kmeans_cluster", "fraction").rdd.collectAsMap()

25/02/07 09:09:12 ERROR Executor: Exception in task 0.0 in stage 16.0 (TID 244)]
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (StandardScalerModel$$$Lambda$3110/0x00000008413ef840: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:217)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage8.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage8.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sq

Py4JJavaError: An error occurred while calling o289.javaToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 16.0 failed 1 times, most recent failure: Lost task 0.0 in stage 16.0 (TID 244) (10.202.33.187 executor driver): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (StandardScalerModel$$$Lambda$3110/0x00000008413ef840: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:217)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage8.hashAgg_doAggregateWithKeys_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.Generated

In [ ]:
binds_0_sampled = binds0.sampleBy("kmeans_cluster", fractions, seed=42)

balanced_data = binds1.unionByName(binds_0_sampled.select(binds1.columns))